**Part 2 · LangChain** · step 13 of 25  
[← 12 Agent structured output](12_agent_structured_output.ipynb) · [🏠 Roadmap](../README.md) · [14 Human-in-the-loop →](14_human_in_the_loop.ipynb)

# 13 · Middleware — Summarization & PII

**Middleware** hooks into the agent loop so you can change behaviour *without* rewriting the agent:

```
before_agent → [ before_model → MODEL → after_model → tools ]* → after_agent
```

Pass them as a list: `create_agent(..., middleware=[A(), B()])` — they run in order.

This notebook covers two built-ins:
| Middleware | Problem it solves |
|---|---|
| `SummarizationMiddleware` | Long chats overflow the context window / cost too much → old messages are summarised |
| `PIIMiddleware` | Personal data (emails, cards, IPs…) should not reach the LLM → redact / mask / hash / block |

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, PIIMiddleware
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver


def count_tokens(messages) -> int:
    """Rough token estimate for display only (≈ 4 characters per token)."""
    return sum(len(str(m.content)) for m in messages) // 4

## 1. Summarization — `trigger` and `keep`

- `trigger` — **when** to summarise: `("messages", N)`, `("tokens", N)` or `("fraction", f)` of the model's context window.
- `keep` — **how much recent history** to keep verbatim after summarising (same units).
- `model` — the (cheap) model used to write the summary.

### a) Message-count trigger
Summarise once there are 10 messages, keep the last 4. Watch the message count drop.

In [ ]:
agent = create_agent(
    model="gpt-4o",
    checkpointer=InMemorySaver(),  # summarisation only matters when history persists
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ],
)
config = {"configurable": {"thread_id": "summary-messages"}}

for q in ["What is 2+2?", "What is 10*5?", "What is 100/4?", "What is 15-7?", "What is 3*3?", "What is 4*4?"]:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"{q:<15} -> {len(response['messages'])} messages in state")

### b) Token trigger
A tool with a long output fills the context quickly. Summarise above ~550 tokens, keep ~200.

In [ ]:
@tool
def search_hotels(city: str) -> str:
    """Search hotels in a city. Returns a long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent = create_agent(
    model="gpt-4o",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(model="gpt-4o-mini", trigger=("tokens", 550), keep=("tokens", 200)),
    ],
)
config = {"configurable": {"thread_id": "summary-tokens"}}

for city in ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]:
    response = agent.invoke({"messages": [HumanMessage(content=f"Find hotels in {city}")]}, config=config)
    print(f"{city}: ~{count_tokens(response['messages'])} tokens, {len(response['messages'])} messages")

### c) Fraction trigger
Fraction of the model's context window (gpt-4o-mini ≈ 128k tokens).
The values below are tiny on purpose so the summary fires during the demo.

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% of 128k ≈ 640 tokens
            keep=("fraction", 0.002),     # 0.2% of 128k ≈ 256 tokens
        ),
    ],
)
config = {"configurable": {"thread_id": "summary-fraction"}}

for city in ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]:
    response = agent.invoke({"messages": [HumanMessage(content=f"Hotels in {city}")]}, config=config)
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens ({tokens / 128_000:.4%}), {len(response['messages'])} msgs")

## 2. PII Middleware

`PIIMiddleware(pii_type, strategy=..., apply_to_input=True)`

| Strategy | Result for `test@mail.com` / `4111111111111111` |
|---|---|
| `redact` | `[REDACTED_EMAIL]` |
| `mask` | keeps only the last digits, e.g. `************1111` |
| `hash` | a deterministic hash (same input → same token) |
| `block` | raise an error, the request is refused |

Built-in types: `email`, `credit_card`, `ip`, `mac_address`, `url` (or pass your own `detector`).

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
    ],
)
config = {"configurable": {"thread_id": "pii-demo"}}

response = agent.invoke(
    {"messages": [HumanMessage("my email address is test@mail.com and my credit card info is 4111111111111111")]},
    config,
)
# The stored HumanMessage is already sanitised - the LLM never saw the real values
response["messages"][0].content

---
✅ **Done with 13 Middleware: summarization & PII.** Next: [14 Human-in-the-loop →](14_human_in_the_loop.ipynb)  
[🏠 Roadmap](../README.md)